# Regresión Lineal: la recta que la máxima verosimilitud escoge

**Ciencia de Datos, Sección A** · Sesión 8 · 18 de agosto de 2026

Notebook companion de la presentación. Corran cada celda con `Shift+Enter`.

Dependencias: `pip install numpy matplotlib scikit-learn`

Igual que el jueves: primero lo derivamos y lo implementamos, y hasta el final llamamos a `sklearn`.

## 1. El problema: predecir un número

El modelo lineal con $n$ observaciones y $d$ features:

$$y = Xw + \varepsilon$$

Cada peso $w_j$ es cuánto cambia $y$ si $x_j$ sube una unidad, dejando lo demás fijo. El **intercepto** se obtiene gratis agregando una columna de unos a $X$.

Simulamos datos: como conocemos los pesos verdaderos, podemos verificar que el estimador los recupera.

In [1]:
import numpy as np

rng = np.random.default_rng(42)
n = 200

X = rng.normal(size=(n, 3))
w_real = np.array([2.0, -1.0, 0.5])
y = X @ w_real + 4.0 + rng.normal(0, 0.5, n)   # intercepto real = 4.0

Xb = np.hstack([np.ones((n, 1)), X])   # columna de unos
print(X.shape, Xb.shape, y.shape)

(200, 3) (200, 4) (200,)


## 2. Mínimos cuadrados es MLE

La pérdida de mínimos cuadrados:

$$\mathcal{L}(w) = \sum_{i=1}^n (y_i - x_i^\top w)^2 = \lVert y - Xw \rVert^2$$

**¿De dónde sale el cuadrado?** Si $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$ e independientes, entonces

$$p(y_i \mid x_i, w) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y_i - x_i^\top w)^2}{2\sigma^2}\right)$$

y la log-verosimilitud de las $n$ observaciones es

$$\ell(w) = -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^n (y_i - x_i^\top w)^2$$

El primer término no depende de $w$ y el segundo lleva signo negativo, así que

$$\arg\max_w \ell(w) = \arg\min_w \lVert y - Xw \rVert^2$$

Mínimos cuadrados es **exactamente** el MLE bajo ruido normal. Cambiar el supuesto de ruido cambia la pérdida.

## 3. Ecuaciones normales

Igualando el gradiente a cero:

$$\nabla_w \mathcal{L}(w) = -2X^\top(y - Xw) = 0 \;\Longrightarrow\; X^\top X w = X^\top y$$

En papel eso es $\hat{w} = (X^\top X)^{-1}X^\top y$, pero **nunca inviertan la matriz**: `np.linalg.solve` es más rápido y numéricamente mejor.

In [2]:
w = np.linalg.solve(Xb.T @ Xb, Xb.T @ y)
print(w)                     # ~ [4.0, 2.0, -1.0, 0.5]
print(np.array([4.0, *w_real]))   # los valores reales

# Comparación: inv es peor idea, aunque aquí dé lo mismo
print(np.linalg.inv(Xb.T @ Xb) @ Xb.T @ y)

[ 3.98147188  2.01839496 -1.00379671  0.47416941]
[ 4.   2.  -1.   0.5]
[ 3.98147188  2.01839496 -1.00379671  0.47416941]


In [3]:
import matplotlib.pyplot as plt

AZUL, ROJO, GRIS, LINEA = "#3A6EA5", "#B04A2E", "#75808E", "#E0E4EA"

def eje_limpio(ax):
    ax.grid(color=LINEA, lw=0.6, alpha=0.7)
    ax.spines[["top", "right"]].set_visible(False)

# Una versión 1D del problema, solo para poder DIBUJAR el modelo
rng1 = np.random.default_rng(8)
x1d = rng1.uniform(0, 10, 40)
y1d = 4.0 + 2.0 * x1d + rng1.normal(0, 2.0, 40)

A = np.hstack([np.ones((40, 1)), x1d[:, None]])
w1d = np.linalg.solve(A.T @ A, A.T @ y1d)
pred1d = A @ w1d

fig, ax = plt.subplots(figsize=(7, 3.6))
for xi, yi, pi in zip(x1d, y1d, pred1d):
    ax.plot([xi, xi], [yi, pi], color=GRIS, lw=0.9, zorder=1)
ax.scatter(x1d, y1d, s=22, color=AZUL, zorder=3, label="datos")
xs = np.array([0.0, 10.0])
ax.plot(xs, w1d[0] + w1d[1] * xs, color=ROJO, lw=2, zorder=2,
        label=f"y = {w1d[0]:.2f} + {w1d[1]:.2f}x")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("OLS elige la recta que minimiza la suma de estos segmentos al cuadrado")
ax.legend(frameon=False)
eje_limpio(ax)
plt.tight_layout()
plt.show()

# Cada segmento gris es un residuo. La pérdida es la suma de sus
# cuadrados, y la recta roja es el argmin: el MLE bajo ruido normal.

/var/folders/b7/rmh5zsw16z55ywlgyckwskqh0000gn/T/ipykernel_87643/1650089053.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Descenso de gradiente: la otra ruta

Si $X^\top X$ no cabe en memoria, bajamos por la pendiente. Mismo destino, camino distinto. En deep learning (semanas 13-15) va a ser la única opción.

In [4]:
w_gd = np.zeros(Xb.shape[1])
lr = 0.05

for _ in range(500):
    grad = -2 / n * Xb.T @ (y - Xb @ w_gd)
    w_gd -= lr * grad

print(w_gd)
print(w)          # la solución cerrada
print(np.abs(w_gd - w).max())

[ 3.98147188  2.01839496 -1.00379671  0.47416941]
[ 3.98147188  2.01839496 -1.00379671  0.47416941]
1.7763568394002505e-15


In [5]:
# Prueben otros learning rates: muy grande diverge, muy chico no llega
for lr in [0.001, 0.05, 1.2]:
    v = np.zeros(Xb.shape[1])
    for _ in range(500):
        v -= lr * (-2 / n * Xb.T @ (y - Xb @ v))
    print(lr, np.round(v, 3))

0.001 [ 2.541  1.246 -0.672  0.194]
0.05 [ 3.981  2.018 -1.004  0.474]
1.2 [-2.95442312e+99  2.85650552e+99  2.77435158e+99 -2.09871842e+98]


In [6]:
def perdida(w0, w1):
    return ((y1d - w0 - w1 * x1d) ** 2).mean()

def camino_gd(lr, pasos):
    v = np.zeros(2)
    tray = [v.copy()]
    for _ in range(pasos):
        v = v - lr * (-2 / len(y1d) * A.T @ (y1d - A @ v))
        tray.append(v.copy())
    return np.array(tray)

g0 = np.linspace(-1, 9, 100)
g1 = np.linspace(-0.5, 4.5, 100)
Z = np.array([[perdida(a, b) for a in g0] for b in g1])

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6), layout="constrained")

ax = axes[0]
ax.contour(g0, g1, Z, levels=25, colors=GRIS, linewidths=0.5)
t_ok = camino_gd(0.02, 400)
t_lento = camino_gd(0.002, 400)
ax.plot(t_ok[:, 0], t_ok[:, 1], lw=2.2, color=AZUL, label="lr = 0.02")
ax.plot(t_lento[:, 0], t_lento[:, 1], marker="o", ms=2.5, lw=1.0,
        color=ROJO, label="lr = 0.002")
ax.scatter(*t_lento[-1], s=45, color=ROJO, zorder=6)
ax.annotate("0.002: tras 400 pasos\naún no llega",
            xy=t_lento[-1], xytext=(4.5, 0.4), fontsize=8, color=ROJO,
            arrowprops=dict(arrowstyle="->", color=ROJO, lw=1))
ax.scatter(*w1d, marker="*", s=160, color=AZUL, zorder=5,
           label="solución cerrada")
ax.set_xlabel("intercepto")
ax.set_ylabel("pendiente")
ax.set_title("El descenso sobre la superficie de pérdida")
ax.legend(frameon=False, fontsize=8, loc="upper left")

ax = axes[1]
for lr, color, ls in [(0.002, ROJO, "-"), (0.02, AZUL, "-"),
                      (0.058, GRIS, "--")]:
    t = camino_gd(lr, 60)
    ax.plot([perdida(a, b) for a, b in t], color=color, ls=ls, lw=1.8,
            label=f"lr = {lr}")
ax.set_yscale("log")
ax.set_xlabel("iteración")
ax.set_ylabel("MSE (escala log)")
ax.set_title("Muy chico no llega; muy grande explota")
ax.legend(frameon=False, fontsize=8)

for ax in axes:
    eje_limpio(ax)
plt.show()

# Ambos caminos suben rápido la pendiente y luego avanzan a lo largo
# del valle alargado y en diagonal: la marca de features sin escalar.
# Mismo destino que np.linalg.solve, pero a pie.

/var/folders/b7/rmh5zsw16z55ywlgyckwskqh0000gn/T/ipykernel_87643/2911478589.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Evaluar honestamente: $R^2$ y residuos

$R^2 = 1$ es predicción perfecta y $R^2 = 0$ es igual que predecir la media. En datos de prueba puede ser **negativo**. Sube siempre al agregar features, aunque sean ruido puro: por eso nunca se juzga un modelo con el $R^2$ de entrenamiento.

In [7]:
pred = Xb @ w
resid = y - pred

ss_res = (resid ** 2).sum()
ss_tot = ((y - y.mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot
print("R2:", round(r2, 4))
print("MSE:", round((resid ** 2).mean(), 4))

R2: 0.9499
MSE: 0.2566


In [8]:
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.scatter(pred, resid, s=14, color=AZUL, alpha=0.7)
ax.axhline(0, color=GRIS, lw=1.2)
ax.set_xlabel("predicción")
ax.set_ylabel("residuo")
ax.set_title("Residuos: buscamos una nube sin forma")
eje_limpio(ax)
plt.tight_layout()
plt.show()

/var/folders/b7/rmh5zsw16z55ywlgyckwskqh0000gn/T/ipykernel_87643/3924486060.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Qué buscar en ese gráfico:**

- Una **curva**: la relación no era lineal, falta un término
- Un **embudo** que se abre: la varianza no es constante
- Puntos **muy lejos**: outliers, que el cuadrado castiga con fuerza

Y el pecado clásico: **extrapolar**. Un modelo ajustado con casas de 80 a 200 m² no sabe nada de una de 600 m². La recta se extiende felizmente hasta el infinito; los datos no.

In [9]:
rng_p = np.random.default_rng(3)
xs_ = rng_p.uniform(0, 10, 150)

casos = [
    ("curva: faltó un término", 0.4 * (xs_ - 5) ** 2 - 3 + rng_p.normal(0, 1, 150)),
    ("embudo: varianza no constante", rng_p.normal(0, 1, 150) * (0.3 + 0.4 * xs_)),
    ("outliers: el cuadrado los castiga", np.where(rng_p.random(150) < 0.03,
                                                   rng_p.normal(0, 12, 150),
                                                   rng_p.normal(0, 1, 150))),
]

fig, axes = plt.subplots(1, 3, figsize=(10, 3.0), layout="constrained")
for ax, (nombre, res) in zip(axes, casos):
    ax.scatter(xs_, res, s=10, color=AZUL, alpha=0.7)
    ax.axhline(0, color=GRIS, lw=1)
    ax.set_title(nombre, fontsize=10)
    ax.set_xlabel("predicción")
    eje_limpio(ax)
axes[0].set_ylabel("residuo")
fig.suptitle("Los tres patrones que delatan al modelo en el gráfico de residuos")
plt.show()

/var/folders/b7/rmh5zsw16z55ywlgyckwskqh0000gn/T/ipykernel_87643/463958237.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Cuando $X^\top X$ se porta mal

Colinealidad: dos features casi redundantes. El síntoma es siempre el mismo, pesos enormes de signo arbitrario que se cancelan entre sí.

In [10]:
X_col = np.hstack([X, (X[:, [0]] + rng.normal(0, 0.01, (n, 1)))])
Xb_col = np.hstack([np.ones((n, 1)), X_col])

w_col = np.linalg.solve(Xb_col.T @ Xb_col, Xb_col.T @ y)
print(np.round(w_col, 2))
print("norma de los pesos:", round(np.linalg.norm(w_col), 2))
print("condición de X'X:", f"{np.linalg.cond(Xb_col.T @ Xb_col):.1e}")

[ 3.98  0.68 -1.    0.48  1.34]
norma de los pesos: 4.4
condición de X'X: 4.1e+04


In [11]:
# El reparto entre los gemelos es arbitrario: reajustar con menos
# datos (una submuestra de 60) lo hace evidente
rng_c = np.random.default_rng(5)
s = rng_c.integers(0, n, 60)
w_sub = np.linalg.solve(Xb_col[s].T @ Xb_col[s], Xb_col[s].T @ y[s])

nombres = ["intercepto", "w1", "w2", "w3", "w4 (copia de w1)"]
reales = [4.0, 2.0, -1.0, 0.5, 0.0]

fig, ax = plt.subplots(figsize=(7.5, 3.4))
pos_x = np.arange(len(w_col))
ax.bar(pos_x - 0.27, reales, width=0.27, color=GRIS, alpha=0.8,
       label="valor real")
ax.bar(pos_x, w_col, width=0.27, color=AZUL, alpha=0.85,
       label="OLS, n = 200")
ax.bar(pos_x + 0.27, w_sub, width=0.27, color=ROJO, alpha=0.85,
       label="OLS, submuestra de 60")
ax.axhline(0, color=GRIS, lw=1)
ax.set_xticks(pos_x, nombres, fontsize=8)
ax.set_ylabel("peso")
ax.set_title("Colinealidad: el reparto entre los gemelos es arbitrario e inestable")
ax.legend(frameon=False, fontsize=8)
eje_limpio(ax)
plt.tight_layout()
plt.show()

# w1 y su copia solo tienen que SUMAR ~2: cada muestra elige un
# reparto distinto, y con pocos datos los gemelos se disparan en
# direcciones opuestas. Ese es el síntoma.

/var/folders/b7/rmh5zsw16z55ywlgyckwskqh0000gn/T/ipykernel_87643/2255381713.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Ridge: penalizar los pesos grandes

$$\mathcal{L}_{\text{ridge}}(w) = \lVert y - Xw \rVert^2 + \lambda \lVert w \rVert^2 \qquad \hat{w}_{\text{ridge}} = (X^\top X + \lambda I)^{-1}X^\top y$$

**Ridge es MAP.** Con prior $w \sim \mathcal{N}(0, \tau^2 I)$:

$$\log p(w \mid X, y) = -\frac{1}{2\sigma^2}\lVert y - Xw\rVert^2 - \frac{1}{2\tau^2}\lVert w \rVert^2 + C$$

Maximizar el posterior es minimizar $\lVert y - Xw\rVert^2 + \lambda\lVert w\rVert^2$ con $\lambda = \sigma^2/\tau^2$. La regularización no es un truco: es una creencia previa escrita como penalización.

In [12]:
def ridge(Xb, y, lam):
    P = lam * np.eye(Xb.shape[1])
    P[0, 0] = 0.0        # no penalizar el intercepto
    return np.linalg.solve(Xb.T @ Xb + P, Xb.T @ y)

for lam in [0.0, 0.1, 1.0, 10.0, 100.0]:
    wr = ridge(Xb_col, y, lam)
    print(lam, np.round(wr, 2), "norma:", round(np.linalg.norm(wr), 2))

0.0 [ 3.98  0.68 -1.    0.48  1.34] norma: 4.4
0.1 [ 3.98  0.98 -1.    0.48  1.04] norma: 4.37
1.0 [ 3.98  1.   -1.    0.47  1.01] norma: 4.37
10.0 [ 3.99  0.98 -0.95  0.44  0.98] norma: 4.35
100.0 [ 4.01  0.8  -0.63  0.26  0.8 ] norma: 4.22


In [13]:
lams_path = np.logspace(-3, 3, 40)
W = np.array([ridge(Xb_col, y, lam) for lam in lams_path])

fig, ax = plt.subplots(figsize=(7.5, 3.4))
estilos = [("intercepto", GRIS, ":"), ("w1", AZUL, "-"), ("w2", AZUL, "--"),
           ("w3", AZUL, "-."), ("w4 (copia)", ROJO, "-")]
for j, (nombre, color, ls) in enumerate(estilos):
    ax.plot(lams_path, W[:, j], color=color, ls=ls, lw=1.8, label=nombre)
ax.set_xscale("log")
ax.set_xlabel("lambda (escala log)")
ax.set_ylabel("peso")
ax.set_title("El camino de ridge: los gemelos se reconcilian y todo se encoge")
ax.legend(frameon=False, fontsize=8, ncols=2)
eje_limpio(ax)
plt.tight_layout()
plt.show()

# Con lambda pequeño, el reparto entre w1 y su copia es el que dictó
# el azar de la muestra. Ridge los iguala (~1 y ~1: la división pareja
# del efecto real de 2) y con lambda enorme aplasta todo hacia el
# prior: cero.

/var/folders/b7/rmh5zsw16z55ywlgyckwskqh0000gn/T/ipykernel_87643/1821212574.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Sesgo y varianza

$$\mathbb{E}[(y - \hat{f}(x))^2] = \text{sesgo}^2 + \text{varianza} + \sigma^2$$

- **Sesgo**: error por asumir una forma equivocada (modelo muy rígido)
- **Varianza**: cuánto cambia el modelo si cambiamos la muestra (modelo muy sensible)
- **Ruido**: lo que nadie puede predecir, ni con datos infinitos

Subir $\lambda$ introduce sesgo a propósito para comprar una reducción mayor de varianza. Midamos la varianza directamente: reajustemos el modelo sobre muchas muestras distintas y veamos cuánto se mueven los pesos.

In [14]:
def pesos_en_muestras(lam, repeticiones=40, m=60):
    salidas = []
    r = np.random.default_rng(1)
    for _ in range(repeticiones):
        s = r.integers(0, n, m)              # muestra con reemplazo
        salidas.append(ridge(Xb_col[s], y[s], lam))
    return np.array(salidas)

for lam in [0.0, 0.1, 1.0, 10.0]:
    P = pesos_en_muestras(lam)
    print("lambda", lam, "| desviación media de los pesos:",
          round(P.std(axis=0).mean(), 3))

lambda 0.0 | desviación media de los pesos: 2.567
lambda 0.1 | desviación media de los pesos: 0.113
lambda 1.0 | desviación media de los pesos: 0.059
lambda 10.0 | desviación media de los pesos: 0.055


In [15]:
lams_v = [0.0, 0.1, 1.0, 10.0]
muestras_w1 = [pesos_en_muestras(lam)[:, 1] for lam in lams_v]

fig, ax = plt.subplots(figsize=(7, 3.4))
bp = ax.boxplot(muestras_w1, patch_artist=True)
for caja in bp["boxes"]:
    caja.set(facecolor=AZUL, alpha=0.35)
for mediana in bp["medians"]:
    mediana.set(color=ROJO, lw=1.5)
ax.set_xticks(range(1, len(lams_v) + 1), [f"lambda = {l}" for l in lams_v])
ax.set_ylabel("w1 en 40 remuestreos")
ax.set_title("La varianza que ridge compra (y el sesgo que cuesta)")
eje_limpio(ax)
plt.tight_layout()
plt.show()

# Con lambda = 0, w1 salta salvajemente entre muestras (varianza alta).
# Al subir lambda la caja se comprime (varianza baja) pero la mediana
# se corre lejos de 2 (sesgo): el intercambio de la fórmula, en vivo.

/var/folders/b7/rmh5zsw16z55ywlgyckwskqh0000gn/T/ipykernel_87643/745956482.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Lo mismo con scikit-learn

`sklearn` agrega la columna de unos por ustedes (`fit_intercept=True`) y tampoco penaliza el intercepto. Su `alpha` es nuestro $\lambda$.

In [16]:
from sklearn.linear_model import LinearRegression, Ridge

ols = LinearRegression().fit(X, y)
rdg = Ridge(alpha=1.0).fit(X, y)

print("sklearn OLS:", np.round(ols.coef_, 3), round(ols.intercept_, 3))
print("a mano     :", np.round(w[1:], 3), round(w[0], 3))
print("sklearn Ridge:", np.round(rdg.coef_, 3), round(rdg.intercept_, 3))

sklearn OLS: [ 2.018 -1.004  0.474] 3.981
a mano     : [ 2.018 -1.004  0.474] 3.981
sklearn Ridge: [ 2.008 -0.998  0.47 ] 3.982


## 10. Ejercicios

Completen donde dice `# ¿Qué va aquí?`.

### Ejercicio 1: OLS a mano vs sklearn

Implementen OLS con las ecuaciones normales sobre datos simulados nuevos y comparen con `LinearRegression`.

In [17]:
rng = np.random.default_rng(0)
n2 = 300
X1 = rng.normal(size=(n2, 4))
w_real1 = np.array([1.5, -2.0, 0.0, 3.0])
y1 = X1 @ w_real1 - 1.0 + rng.normal(0, 1.0, n2)

def ols(X, y):
    # ¿Qué va aquí?
    # 1) agregar la columna de unos
    # 2) resolver con np.linalg.solve (NO con inv)
    pass

# Verificación (descomenten al terminar):
# print(ols(X1, y1))
# print(LinearRegression().fit(X1, y1).intercept_,
#       LinearRegression().fit(X1, y1).coef_)

### Ejercicio 2: un feature de puro ruido

Agreguen una columna sin ninguna señal y observen su peso con OLS y con Ridge. El peso ideal sería exactamente 0. ¿Cuál de los dos estimadores es más honesto sobre lo que no sabe?

In [18]:
ruido = rng.normal(size=(n2, 1))
X2 = np.hstack([X1, ruido])   # el último feature no explica nada

# ¿Qué va aquí?
# a) ajustar OLS sobre X2 e imprimir el peso del feature de ruido
# b) ajustar Ridge (lam=10) e imprimir ese mismo peso
# c) repetir todo con otra semilla: ¿cuál peso se mueve más?
pass

### Ejercicio 3: la curva de $\lambda$

Grafiquen el error de entrenamiento y el de prueba al variar $\lambda$. ¿Dónde está el mínimo de la curva de test? Eso es sesgo-varianza, visto de frente.

In [19]:
corte = 200
Xb2 = np.hstack([np.ones((n2, 1)), X2])
Xtr, ytr = Xb2[:corte], y1[:corte]
Xte, yte = Xb2[corte:], y1[corte:]

lams = np.logspace(-3, 3, 25)
err_train, err_test = [], []

for lam in lams:
    # ¿Qué va aquí?
    # 1) ajustar ridge(Xtr, ytr, lam)
    # 2) guardar el MSE de train y el MSE de test
    pass

# Verificación (descomenten al terminar):
# plt.semilogx(lams, err_train, label="train")
# plt.semilogx(lams, err_test, label="test")
# plt.xlabel("lambda"); plt.ylabel("MSE"); plt.legend(); plt.show()

## Lo esencial de hoy

- El modelo lineal $y = Xw + \varepsilon$; el intercepto es una columna de unos
- **Mínimos cuadrados = MLE** con ruido normal: el cuadrado no es arbitrario
- **Ecuaciones normales**: $X^\top X w = X^\top y$, con `solve`, nunca `inv`
- Evaluar con $R^2$ **y** residuos; extrapolar es el pecado clásico
- **Ridge = MAP** con prior normal sobre los pesos; $\lambda$ es la perilla
- Error $=$ sesgo$^2$ $+$ varianza $+$ ruido: el óptimo está en el medio

**Próxima clase (jueves 20): EDA y Visualización.** Se asigna la HDT 3 (entrega: martes 25 de agosto).